# Instrumental Variables: Classic Returns-to-Education Estimation

**Module 06 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement OLS, IV (Wald), and 2SLS estimators from scratch
2. Demonstrate ability bias in OLS and show that IV corrects for it
3. Test the first stage for instrument relevance
4. Interpret the IV coefficient as a LATE for compliers

## Context

This notebook replicates the core analysis of Card (1995) — estimating the return to education using geographic proximity to a four-year college as an instrument. The key challenge: ability is unobservable and correlated with both education and wages. College proximity is plausibly unrelated to ability but shifts educational attainment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(1995)  # Card (1995)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Data Generating Process

We simulate data with the following structure:

- **Ability** $a_i$ ~ N(0, 1): unobservable, creates endogeneity
- **College nearby** $z_i$ ~ Bernoulli(0.5): the instrument (exogenous)
- **Education** $s_i$: depends on ability and college proximity → endogenous
- **Wages** $w_i$: depends on education, ability, and experience

True causal return to education: **8% per year** (0.08 in log wages)

In [ ]:
N = 3000
TRUE_RETURN = 0.08  # 8% per year of education

# Unobservable ability (creates endogeneity)
ability = np.random.normal(0, 1, N)

# Instrument: college nearby (exogenous)
college_nearby = np.random.binomial(1, 0.5, N)

# Education: affected by ability and college proximity
# True first stage: being near college increases education by 0.5 years
experience = np.random.randint(1, 30, N)
female = np.random.binomial(1, 0.5, N)

education = (10
             + 0.5 * college_nearby   # instrument effect (first stage)
             + 1.5 * ability          # ability drives education (confounder)
             + 0.05 * experience      # experience slightly correlated
             + np.random.normal(0, 1.5, N))
education = education.round().clip(8, 22)  # 8-22 years

# Log wages: depends on education (causal), ability (confounder), experience
log_wage = (1.5
            + TRUE_RETURN * education   # causal return to education
            + 0.15 * ability            # direct ability effect on wages
            + 0.04 * experience
            - 0.001 * experience**2    # diminishing returns to experience
            - 0.15 * female            # gender gap
            + np.random.normal(0, 0.25, N))

df = pd.DataFrame({
    'log_wage': log_wage,
    'education': education,
    'college_nearby': college_nearby,
    'experience': experience,
    'female': female,
    'ability': ability   # normally unobservable — included for illustration
})

print(f"N = {N}, True return to education = {TRUE_RETURN:.2f}")
print(f"\nDescriptive statistics:")
print(df[['log_wage', 'education', 'college_nearby']].describe().round(3))
print(f"\nFirst stage: education near college = {df[df['college_nearby']==1]['education'].mean():.2f}")
print(f"             education far from college = {df[df['college_nearby']==0]['education'].mean():.2f}")

## 2. The OLS Estimate: Biased by Ability

OLS regresses log wage on education and controls, but cannot control for unobserved ability.

In [ ]:
# OLS without ability control (what a researcher actually sees)
model_ols = smf.ols('log_wage ~ education + experience + female', data=df).fit(cov_type='HC1')

# OLS with ability control (unbiased but ability is not observed)
model_oracle = smf.ols('log_wage ~ education + experience + female + ability', data=df).fit(cov_type='HC1')

ols_return = model_ols.params['education']
oracle_return = model_oracle.params['education']

print("Return to Education Estimates:")
print("=" * 55)
print(f"{'Method':<30} {'Estimate':>10} {'SE':>8}")
print("-" * 55)
print(f"{'True effect':<30} {TRUE_RETURN:>10.4f}")
print(f"{'OLS (ability omitted)':<30} {ols_return:>10.4f} {model_ols.bse['education']:>8.4f}")
print(f"{'Oracle OLS (ability known)':<30} {oracle_return:>10.4f} {model_oracle.bse['education']:>8.4f}")
print()
print(f"OLS bias: {ols_return - TRUE_RETURN:+.4f}")
print(f"Direction: {'upward' if ols_return > TRUE_RETURN else 'downward'} bias")
print()
print(f"Ability coefficient in oracle: {model_oracle.params['ability']:.4f}")
print("(Ability has positive direct effect on wages AND education → upward OLS bias)")

## 3. The Wald Estimator

The simplest IV estimator for a binary instrument. Computes the ratio of the reduced form effect to the first stage effect.

In [ ]:
# Wald estimator: no controls
# First stage: effect of instrument on endogenous variable
E_educ_z1 = df[df['college_nearby']==1]['education'].mean()
E_educ_z0 = df[df['college_nearby']==0]['education'].mean()
first_stage_wald = E_educ_z1 - E_educ_z0

# Reduced form: effect of instrument on outcome
E_wage_z1 = df[df['college_nearby']==1]['log_wage'].mean()
E_wage_z0 = df[df['college_nearby']==0]['log_wage'].mean()
reduced_form_wald = E_wage_z1 - E_wage_z0

# IV estimate
wald_iv = reduced_form_wald / first_stage_wald

print("Wald Estimator Decomposition:")
print("=" * 45)
print(f"First stage:   E[educ | Z=1] - E[educ | Z=0] = {E_educ_z1:.3f} - {E_educ_z0:.3f} = {first_stage_wald:.4f}")
print(f"Reduced form:  E[wage | Z=1] - E[wage | Z=0] = {E_wage_z1:.3f} - {E_wage_z0:.3f} = {reduced_form_wald:.4f}")
print(f"Wald IV:       Reduced form / First stage = {reduced_form_wald:.4f} / {first_stage_wald:.4f} = {wald_iv:.4f}")
print(f"\nTrue effect: {TRUE_RETURN:.4f}")
print(f"Wald bias: {wald_iv - TRUE_RETURN:+.4f}")

## 4. 2SLS with Controls

In [ ]:
# Manual 2SLS with controls
# Stage 1: Predict education using instrument + controls
stage1 = smf.ols('education ~ college_nearby + experience + female', data=df).fit()
df['educ_hat'] = stage1.fittedvalues  # predicted education (exogenous part)

# Stage 2: Regress log_wage on predicted education (WRONG SE — for illustration only)
stage2_wrong_se = smf.ols('log_wage ~ educ_hat + experience + female', data=df).fit()

# Correct 2SLS using statsmodels IV2SLS
from statsmodels.sandbox.regression.gmm import IV2SLS

# Endog: [education], instruments: [college_nearby, experience, female]
# The controls (experience, female) are both in X and in Z (as their own instruments)
X_with_const = sm.add_constant(df[['education', 'experience', 'female']])
Z = sm.add_constant(df[['college_nearby', 'experience', 'female']])

iv2sls = IV2SLS(df['log_wage'], X_with_const, Z).fit()

iv_return = iv2sls.params['education']
iv_se = iv2sls.bse['education']
iv_ci = iv2sls.conf_int().loc['education'].values

# First stage statistics
f_stat = stage1.fvalue
instrument_coef = stage1.params['college_nearby']
instrument_t = stage1.tvalues['college_nearby']

print("2SLS Results:")
print("=" * 55)
print(f"IV estimate: {iv_return:.4f}")
print(f"SE:          {iv_se:.4f}")
print(f"95% CI:      [{iv_ci[0]:.4f}, {iv_ci[1]:.4f}]")
print(f"True effect: {TRUE_RETURN:.4f}")
print()
print("First Stage Diagnostics:")
print(f"  Instrument coef: {instrument_coef:.4f} (t = {instrument_t:.2f})")
print(f"  First stage F:   {f_stat:.2f}")
status = 'STRONG' if f_stat > 10 else 'WEAK'
print(f"  Instrument strength: {status} (F {'>' if f_stat > 10 else '<'} 10)")

## 5. Comparison of All Estimates

In [ ]:
# Compare all estimates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: forest plot
estimates = [
    ('True Effect', TRUE_RETURN, None, None),
    ('Oracle OLS\n(ability known)', oracle_return,
     oracle_return - 1.96*model_oracle.bse['education'],
     oracle_return + 1.96*model_oracle.bse['education']),
    ('OLS\n(ability omitted)', ols_return,
     ols_return - 1.96*model_ols.bse['education'],
     ols_return + 1.96*model_ols.bse['education']),
    ('Wald IV\n(no controls)', wald_iv, None, None),
    ('2SLS\n(with controls)', iv_return, iv_ci[0], iv_ci[1]),
]

ax = axes[0]
colors = ['green', 'steelblue', 'red', 'purple', 'darkorange']
for i, (label, est, lo, hi) in enumerate(estimates):
    ax.scatter(est, i, color=colors[i], s=80, zorder=5)
    if lo is not None and hi is not None:
        ax.plot([lo, hi], [i, i], color=colors[i], linewidth=2.5)
    ax.text(est + 0.002, i + 0.12, f'{est:.4f}', fontsize=9, color=colors[i])

ax.axvline(TRUE_RETURN, color='green', linestyle=':', alpha=0.5)
ax.set_yticks(range(len(estimates)))
ax.set_yticklabels([e[0] for e in estimates], fontsize=9)
ax.set_xlabel('Return to Education (per year)')
ax.set_title('Estimates: OLS vs IV\n(OLS biased upward; IV corrects)')
ax.set_xlim(0.07, 0.12)

# Right: first stage plot
ax2 = axes[1]
for side, label, color in [(0, 'No college nearby', 'steelblue'), (1, 'College nearby', 'darkorange')]:
    subset = df[df['college_nearby'] == side]
    ax2.hist(subset['education'], bins=15, alpha=0.5, color=color,
             label=f'{label} (mean={subset["education"].mean():.1f})', density=True)

ax2.axvline(df[df['college_nearby']==0]['education'].mean(),
            color='steelblue', linestyle='--', linewidth=2)
ax2.axvline(df[df['college_nearby']==1]['education'].mean(),
            color='darkorange', linestyle='--', linewidth=2)
ax2.set_xlabel('Years of Education')
ax2.set_ylabel('Density')
ax2.set_title(f'First Stage: College Proximity → Education\n(shift = {first_stage_wald:.2f} years, F = {f_stat:.1f})')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. LATE vs ATE: Who Are the Compliers?

The IV estimate is a LATE — the return to education for those who are induced to get more education by being near a college.

In [ ]:
# Approximate complier characteristics
# Compliers = those for whom college_nearby shifts education from <threshold to >=threshold

# Proxy: units who are near college and have education near the margin
# (This is an approximation; true complier identification requires causal models)

first_stage_effect = first_stage_wald  # ~0.5 years

print("LATE Interpretation:")
print("=" * 55)
print(f"IV estimates the return to education for COMPLIERS:")
print(f"  → People who got more education because they lived")
print(f"    near a college")
print(f"  → College proximity shifted their education by ~{first_stage_effect:.2f} years")
print()
print(f"Who are compliers likely to be?")
print(f"  - From families where distance/cost is a binding constraint")
print(f"  - Moderate ability (high ability = always go regardless)")
print(f"  - Lower income (cost-sensitive)")
print()
print(f"2SLS estimate (LATE): {iv_return:.4f}")
print(f"OLS estimate (ATE?):  {ols_return:.4f} (biased)")
print(f"True ATE:             {TRUE_RETURN:.4f}")
print()
print("The IV estimate is the return to education for marginal students.")
print("This is the policy-relevant effect for scholarships/access programs.")

## Self-Check

1. Increase the ability effect on education from `1.5` to `3.0`. How does the OLS bias change? Does the IV estimate also change?

2. Reduce the first stage effect from `0.5` to `0.1` (weaker instrument). What happens to the IV estimate's variance? What happens to the F-statistic?

3. Add a direct effect of college proximity on wages: `log_wage += 0.02 * college_nearby`. This violates the exclusion restriction. How much does this bias the IV estimate?

4. Include ability in the OLS regression (oracle model). Does the IV estimate also change when you control for ability? Why or why not?

---
**Next:** `02_weak_instruments.ipynb` — Diagnosing and handling weak instruments